# Machine Learning Models with Scikit-Learn

Covers 4 core algorithms:

| # | Algorithm | Type | Predicts |
|---|---|---|---|
| 1 | **Linear Regression** | Regression | Continuous number (marks, price) |
| 2 | **Logistic Regression** | Classification | Category (Pass/Fail, Yes/No) |
| 3 | **KNN Classifier** | Classification | Category based on nearest neighbors |
| 4 | **Decision Tree** | Classification | Category using if/else rules |

## 1 — Linear Regression

Finds the best-fit **straight line** through data points.
- Formula: `y = m*x + b`  (m = slope, b = intercept)
- Use when: output is a **continuous number** (score, salary, price)
- The model learns `m` and `b` from training data, then predicts `y` for new `x`

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# X must be 2D (list of lists) — each inner list is one sample's features
# Even for 1 feature, sklearn requires shape (n_samples, n_features)
X = [[1], [2], [3], [4], [5]]  # study hours
y = [40, 50, 65, 75, 90]       # test scores

model = LinearRegression()  # create the model object (no learning yet)

# fit() is where actual learning happens — finds best m and b
model.fit(X, y)

# coef_ = slope (m): how much score increases per extra hour of study
# intercept_ = b: predicted score when study hours = 0
print('=== What the Model Learned ===')
print(f'Slope (coef_)     : {model.coef_[0]:.2f}  → each extra study hour adds ~{model.coef_[0]:.1f} marks')
print(f'Intercept         : {model.intercept_:.2f} → base score with 0 hours')
print(f'Line equation     : score = {model.coef_[0]:.2f} × hours + {model.intercept_:.2f}')

# predict() applies the learned formula to new unseen inputs
test_hours = [1, 3, 6, 10]  # hours we want to predict scores for
predictions = model.predict([[h] for h in test_hours])  # must be 2D

print('\n=== Predictions ===')
for h, p in zip(test_hours, predictions):
    print(f'  {h} hours studied  →  predicted score: {p:.1f}')

# --- Model Evaluation ---
y_pred_train = model.predict(X)  # predict on training data to check fit

# R² score: how well the line fits the data (1.0 = perfect, 0 = no fit)
r2 = r2_score(y, y_pred_train)

# MAE: average absolute difference between predicted and actual values
mae = mean_absolute_error(y, y_pred_train)

print('\n=== Model Evaluation (on training data) ===')
print(f'R² Score (fit quality) : {r2:.4f}  (1.0 = perfect fit)')
print(f'MAE (avg error)        : {mae:.2f} marks')

# Show actual vs predicted side by side
print('\n=== Actual vs Predicted (Training Data) ===')
df_results = pd.DataFrame({'Hours': [x[0] for x in X], 'Actual': y, 'Predicted': y_pred_train.round(1)})
print(df_results.to_string(index=False))

=== What the Model Learned ===
Slope (coef_)     : 12.50  → each extra study hour adds ~12.5 marks
Intercept         : 26.50 → base score with 0 hours
Line equation     : score = 12.50 × hours + 26.50

=== Predictions ===
  1 hours studied  →  predicted score: 39.0
  3 hours studied  →  predicted score: 64.0
  6 hours studied  →  predicted score: 101.5
  10 hours studied  →  predicted score: 151.5

=== Model Evaluation (on training data) ===
R² Score (fit quality) : 0.9952  (1.0 = perfect fit)
MAE (avg error)        : 1.20 marks

=== Actual vs Predicted (Training Data) ===
 Hours  Actual  Predicted
     1      40       39.0
     2      50       51.5
     3      65       64.0
     4      75       76.5
     5      90       89.0


## 2 — Logistic Regression

Despite the name, it's a **classifier** — not a regression model.
- Predicts **probability** of belonging to a class, then picks the class
- Output is always 0 or 1 (binary classification)
- Uses a **sigmoid curve** instead of a straight line
- Use when: output is **two categories** (Pass/Fail, Spam/Not Spam, Yes/No)

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X = [[1], [2], [3], [4], [5]]  # hours studied (2D required)
y = [0,   0,   1,   1,   1]   # 0 = Fail, 1 = Pass

# max_iter: max optimization steps (increase if model shows ConvergenceWarning)
model = LogisticRegression(max_iter=200)

# fit() learns the decision boundary — the point where class switches 0→1
model.fit(X, y)

# predict_proba() returns [P(class=0), P(class=1)] for each input
# This shows the model's confidence, not just the final label
print('=== Prediction Probabilities ===')
print(f'{"Hours":<8} {"P(Fail)":<12} {"P(Pass)":<12} {"Result"}')
print('-' * 42)
for h in [1, 2, 3, 4, 5, 6]:
    proba = model.predict_proba([[h]])[0]   # [P(0), P(1)]
    label = model.predict([[h]])[0]         # final predicted class
    outcome = 'PASS' if label == 1 else 'FAIL'
    print(f'{h:<8} {proba[0]:<12.3f} {proba[1]:<12.3f} {outcome}')

# accuracy_score: fraction of correct predictions (correct / total)
y_pred = model.predict(X)
acc = accuracy_score(y, y_pred)
print(f'\n=== Accuracy on Training Data: {acc*100:.0f}% ===')

# classification_report: detailed breakdown — precision, recall, F1 per class
# precision = of all predicted Pass, how many were really Pass?
# recall    = of all actual Pass, how many did we catch?
# F1        = balance between precision and recall
print('\n=== Classification Report ===')
print(classification_report(y, y_pred, target_names=['Fail', 'Pass']))

=== Prediction Probabilities ===
Hours    P(Fail)      P(Pass)      Result
------------------------------------------
1        0.816        0.184        FAIL
2        0.608        0.392        FAIL
3        0.353        0.647        PASS
4        0.161        0.839        PASS
5        0.063        0.937        PASS
6        0.023        0.977        PASS

=== Accuracy on Training Data: 100% ===

=== Classification Report ===
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00         2
        Pass       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



## 3 — K-Nearest Neighbors (KNN)

Classifies by looking at the **K closest training examples** and taking a majority vote.
- No real "training" — just memorizes all data points
- `n_neighbors=3` → looks at 3 nearest points and votes
- Use when: similar items should have similar labels
- **Sensitive to scale** — features with larger ranges dominate distance calculation

In [4]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Training data: [weight_grams, size_cm]
X = [
    [180, 7.0],  # Apple
    [200, 7.5],  # Apple
    [220, 7.8],  # Apple
    [290, 8.5],  # Orange
    [320, 9.0],  # Orange
    [360, 9.5],  # Orange
]
y = [0, 0, 0, 1, 1, 1]  # 0 = Apple, 1 = Orange

# n_neighbors=3: when predicting, look at the 3 closest training samples
# majority class among those 3 neighbors wins
model = KNeighborsClassifier(n_neighbors=3)

# fit() just stores the training data — no computation yet
model.fit(X, y)

# Test with new fruit samples
test_fruits = [
    [190, 7.2],  # should be Apple
    [310, 9.1],  # should be Orange
    [250, 8.2],  # borderline
]

print('=== KNN Predictions (n_neighbors=3) ===')
print(f'{"Weight(g)":<12} {"Size(cm)":<12} {"Prediction":<12} {"Confidence"}')
print('-' * 50)
for fruit in test_fruits:
    label = model.predict([fruit])[0]          # predicted class (0 or 1)
    proba = model.predict_proba([fruit])[0]    # [P(Apple), P(Orange)]
    name  = 'Apple' if label == 0 else 'Orange'
    conf  = max(proba) * 100                   # confidence = highest probability
    print(f'{fruit[0]:<12} {fruit[1]:<12} {name:<12} {conf:.0f}%')

# kneighbors() shows WHICH neighbors were used for a prediction
distances, indices = model.kneighbors([[250, 8.2]])
print(f'\n=== Neighbors used for [250g, 8.2cm] ===')
labels_map = {0: 'Apple', 1: 'Orange'}
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f'  Neighbor {i+1}: {X[idx]} → {labels_map[y[idx]]}  (distance: {dist:.2f})')

# Accuracy on training data
y_pred = model.predict(X)
print(f'\nAccuracy on training data: {accuracy_score(y, y_pred)*100:.0f}%')

# Effect of different K values
print('\n=== Effect of n_neighbors (K) on training accuracy ===')
for k in [1, 2, 3, 4, 5]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    acc = accuracy_score(y, m.predict(X))
    # k=1 always gets 100% on training (memorizes perfectly) — usually overfits
    note = '← overfits (memorizes)' if k == 1 else ''
    print(f'  K={k}: {acc*100:.0f}%  {note}')

=== KNN Predictions (n_neighbors=3) ===
Weight(g)    Size(cm)     Prediction   Confidence
--------------------------------------------------
190          7.2          Apple        100%
310          9.1          Orange       100%
250          8.2          Apple        67%

=== Neighbors used for [250g, 8.2cm] ===
  Neighbor 1: [220, 7.8] → Apple  (distance: 30.00)
  Neighbor 2: [290, 8.5] → Orange  (distance: 40.00)
  Neighbor 3: [200, 7.5] → Apple  (distance: 50.00)

Accuracy on training data: 100%

=== Effect of n_neighbors (K) on training accuracy ===
  K=1: 100%  ← overfits (memorizes)
  K=2: 100%  
  K=3: 100%  
  K=4: 100%  
  K=5: 100%  


## 4 — Decision Tree Classifier

Builds a tree of **if/else questions** to split data into classes.
- Human-readable — you can visualize and interpret exactly what rules it learned
- No need to scale features (unlike KNN or SVM)
- Prone to **overfitting** on small datasets — `max_depth` controls this
- Use when: you need explainability, or features have clear thresholds

In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

# Features: [sweetness (1-10), size_cm]
X = [
    [7, 2],   # Apple — sweet, small
    [8, 3],   # Apple — sweet, small
    [4, 8],   # Orange — less sweet, big
    [3, 9],   # Orange — less sweet, big
    [9, 2.5], # Apple
    [2, 8.5], # Orange
]
y = [0, 0, 1, 1, 0, 1]  # 0 = Apple, 1 = Orange

# max_depth limits how deep the tree grows — prevents overfitting
# Without max_depth, tree memorizes every training point (overfit)
model = DecisionTreeClassifier(max_depth=3, random_state=42)

# fit() builds the if/else tree from training data
model.fit(X, y)

# export_text() prints the learned decision rules in human-readable form
# feature_names maps column index to a readable label
tree_rules = export_text(model, feature_names=['sweetness', 'size_cm'])
print('=== Learned Decision Tree Rules ===')
print(tree_rules)

# feature_importances_: how much each feature contributed to splits
# Higher value = more important for the decisions
print('=== Feature Importances ===')
for name, importance in zip(['sweetness', 'size_cm'], model.feature_importances_):
    bar = '█' * int(importance * 20)  # visual bar
    print(f'  {name:<12}: {importance:.3f}  {bar}')

# Predict on new samples
test_items = [
    [8.0, 2.5],  # sweet + small → Apple
    [3.0, 9.0],  # not sweet + big → Orange
    [6.0, 5.0],  # borderline
]
print('\n=== Predictions ===')
print(f'{"Sweetness":<12} {"Size(cm)":<12} {"Prediction"}')
print('-' * 36)
for item in test_items:
    label = model.predict([item])[0]
    name  = 'Apple' if label == 0 else 'Orange'
    print(f'{item[0]:<12} {item[1]:<12} {name}')

# tree_depth shows how deep the tree actually grew
print(f'\nTree depth used: {model.get_depth()}  (max allowed: {model.max_depth})')
print(f'Total leaf nodes: {model.get_n_leaves()}  (each leaf = one final decision)')

y_pred = model.predict(X)
print(f'Accuracy on training data: {accuracy_score(y, y_pred)*100:.0f}%')

=== Learned Decision Tree Rules ===
|--- sweetness <= 5.50
|   |--- class: 1
|--- sweetness >  5.50
|   |--- class: 0

=== Feature Importances ===
  sweetness   : 1.000  ████████████████████
  size_cm     : 0.000  

=== Predictions ===
Sweetness    Size(cm)     Prediction
------------------------------------
8.0          2.5          Apple
3.0          9.0          Orange
6.0          5.0          Apple

Tree depth used: 1  (max allowed: 3)
Total leaf nodes: 2  (each leaf = one final decision)
Accuracy on training data: 100%


## Summary — When to Use Which Model

```
What kind of output do you need?
  │
  ├─ Continuous number (score, price, temperature)
  │    └─► Linear Regression
  │         • coef_ = slope  |  intercept_ = base value
  │         • Evaluate with: R² score, MAE
  │
  └─ Category (class label)
       │
       ├─ Binary (2 classes: Pass/Fail)
       │    └─► Logistic Regression
       │         • predict_proba() = confidence scores
       │         • Evaluate with: accuracy, precision, recall, F1
       │
       ├─ Based on similarity to neighbors
       │    └─► KNN Classifier
       │         • n_neighbors = how many neighbors vote
       │         • Scale features first! (KNN is distance-based)
       │
       └─ Need explainability / if-else rules
            └─► Decision Tree
                 • export_text() shows learned rules
                 • max_depth prevents overfitting
                 • feature_importances_ = which features matter most
```

### Key sklearn pattern (same for all models)
```python
model = SomeModel()        # 1. Create
model.fit(X_train, y_train)  # 2. Train (learn from data)
model.predict(X_test)      # 3. Predict on new data
```

## 5 — Ridge & Lasso Regression (Regularization)

**Problem with plain Linear Regression:** on small or noisy datasets, the model can overfit — fitting noise instead of the real pattern. The solution is **regularization**: add a penalty to the loss function that discourages large coefficients.

| Model | Penalty | Effect | Use When |
|---|---|---|---|
| **Ridge** (L2) | Sum of squared coefficients | Shrinks all coefficients toward 0 | Many small useful features |
| **Lasso** (L1) | Sum of absolute coefficients | Drives some coefficients to exactly 0 (feature selection!) | Sparse features — only a few really matter |

`alpha` controls penalty strength: higher `alpha` = more shrinkage = simpler model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
n = 80
# 6 features — only 2 truly matter; the other 4 are noise
study   = np.random.uniform(1, 10, n)
attend  = np.random.uniform(40, 100, n)
noise1  = np.random.randn(n)
noise2  = np.random.randn(n)
noise3  = np.random.randn(n)
noise4  = np.random.randn(n)

X = np.column_stack([study, attend, noise1, noise2, noise3, noise4])
y = 5 * study + 0.3 * attend + np.random.normal(0, 6, n)   # only study+attend matter

feat_names = ['StudyHrs', 'Attend', 'Noise1', 'Noise2', 'Noise3', 'Noise4']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge(alpha=1)':   Ridge(alpha=1.0),
    'Ridge(alpha=10)':  Ridge(alpha=10.0),
    'Lasso(alpha=0.5)': Lasso(alpha=0.5),
    'Lasso(alpha=2)':   Lasso(alpha=2.0),
}

print(f'{"Model":<22} {"Train R²":>9} {"Test R²":>9} {"MAE":>7}')
print('-' * 50)
coef_table = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    tr2 = r2_score(y_train, m.predict(X_train_s))
    te2 = r2_score(y_test,  m.predict(X_test_s))
    mae = mean_absolute_error(y_test, m.predict(X_test_s))
    print(f'{name:<22} {tr2:>9.4f} {te2:>9.4f} {mae:>7.2f}')
    coef_table[name] = m.coef_

print('\n=== Learned Coefficients (StandardScaled) ===')
coef_df = pd.DataFrame(coef_table, index=feat_names).round(3)
print(coef_df)
print('\n→ Lasso drives Noise coefficients to 0 (automatic feature selection)')
print('→ Ridge shrinks all coefficients but keeps none exactly at 0')

## 6 — Random Forest Classifier

A **Random Forest** builds many Decision Trees on random subsets of data and features, then takes a majority vote. This is called **ensemble learning** — combining weak models to make a stronger one.

| Property | Decision Tree | Random Forest |
|---|---|---|
| Number of trees | 1 | `n_estimators` (default 100) |
| Overfitting | High | Low (averaging reduces variance) |
| Speed | Fast | Slower (trains many trees) |
| Feature importance | Yes | Yes (averaged over all trees) |

`n_estimators` = number of trees. More trees → more stable, but slower.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import make_classification

np.random.seed(42)
X, y = make_classification(
    n_samples=300, n_features=5, n_informative=3,
    n_redundant=1, random_state=42
)
feat_names = ['Feature1', 'Feature2', 'Feature3', 'Feature4', 'Feature5']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Decision Tree (single) vs Random Forest (ensemble of 100 trees)
dt  = DecisionTreeClassifier(random_state=42)
rf  = RandomForestClassifier(n_estimators=100, random_state=42)

dt.fit(X_train, y_train)
rf.fit(X_train, y_train)

print(f'{"Model":<25} {"Train Acc":>10} {"Test Acc":>10}')
print('-' * 47)
for name, m in [('DecisionTree', dt), ('RandomForest(100 trees)', rf)]:
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test,  m.predict(X_test))
    print(f'{name:<25} {tr:>10.4f} {te:>10.4f}')

print('\n=== Random Forest — Classification Report ===')
print(classification_report(y_test, rf.predict(X_test)))

# Feature importances: which features the Random Forest used most across all trees
print('=== Feature Importances (averaged over all 100 trees) ===')
for name, imp in sorted(zip(feat_names, rf.feature_importances_),
                         key=lambda x: -x[1]):
    bar = '█' * int(imp * 40)
    print(f'  {name:<12}: {imp:.3f}  {bar}')

# Effect of n_estimators on test accuracy
print('\n=== Effect of n_estimators on Test Accuracy ===')
for n in [1, 5, 10, 50, 100, 200]:
    m = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f'  n_estimators={n:<5}: {acc:.4f}')

## 7 — Support Vector Machine (SVM)

SVM finds the **maximum-margin hyperplane** that best separates classes.

- The **margin** is the gap between the hyperplane and the nearest data points (called **support vectors**)
- Wider margin → better generalization to unseen data
- **`kernel`** controls the shape of the decision boundary:
  - `'linear'` — straight line/plane (fast, works for linearly separable data)
  - `'rbf'` (Radial Basis Function) — curved boundary (handles non-linear data, default)
- **`C`** (regularization): high C → fits training data tighter (risk of overfit); low C → wider margin, simpler boundary
- **Scale features first** — SVM is distance-based and very sensitive to feature magnitude

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import make_classification

np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=4, n_informative=3,
                            n_redundant=0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# SVM is sensitive to feature scale — always scale before fitting
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Compare kernel types and C values
configs = [
    ('SVM linear C=0.1', SVC(kernel='linear', C=0.1,  random_state=42)),
    ('SVM linear C=1',   SVC(kernel='linear', C=1.0,  random_state=42)),
    ('SVM linear C=10',  SVC(kernel='linear', C=10.0, random_state=42)),
    ('SVM rbf C=0.1',    SVC(kernel='rbf',    C=0.1,  random_state=42)),
    ('SVM rbf C=1',      SVC(kernel='rbf',    C=1.0,  random_state=42)),
    ('SVM rbf C=10',     SVC(kernel='rbf',    C=10.0, random_state=42)),
]

print(f'{"Config":<22} {"Train Acc":>10} {"Test Acc":>10}')
print('-' * 45)
for name, m in configs:
    m.fit(X_train_s, y_train)
    tr = accuracy_score(y_train, m.predict(X_train_s))
    te = accuracy_score(y_test,  m.predict(X_test_s))
    print(f'{name:<22} {tr:>10.4f} {te:>10.4f}')

# Best config detailed report
best_svm = SVC(kernel='rbf', C=1.0, random_state=42)
best_svm.fit(X_train_s, y_train)
print('\n=== SVM rbf C=1 — Classification Report ===')
print(classification_report(y_test, best_svm.predict(X_test_s)))

print(f'Support vectors used: {best_svm.n_support_}  '
      f'(class 0: {best_svm.n_support_[0]}, class 1: {best_svm.n_support_[1]})')
print('→ Support vectors are the training points closest to the decision boundary')
print('→ Only these points affect the model — others can be removed with no change')

## 8 — Underfitting vs Overfitting & Bias-Variance Tradeoff

```
Underfitting (High Bias)         Good Fit              Overfitting (High Variance)
───────────────────────────────────────────────────────────────────────────────────
Train acc: LOW                   Train acc: HIGH        Train acc: VERY HIGH
Test  acc: LOW                   Test  acc: HIGH        Test  acc: LOW
                                                        (gap = overfit signal)

Cause: model too simple          Cause: right fit       Cause: model too complex,
       (max_depth=1, C too low)                                memorised training noise

Fix:   more features, less       ──── ideal ────        Fix: regularize (Ridge/Lasso),
       regularization                                        reduce max_depth,
                                                             more training data
```

**Bias-Variance Tradeoff:**
- **Bias** = how wrong the model is on average (underfitting → high bias)
- **Variance** = how much predictions change across different training sets (overfitting → high variance)
- You cannot minimize both simultaneously — must find the right balance

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=200, n_features=5, n_informative=3,
                            n_redundant=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print('=== Underfitting vs Overfitting (Decision Tree max_depth) ===')
print(f'{"max_depth":<12} {"Train Acc":>10} {"Test Acc":>10} {"Gap":>8}  {"Status"}')
print('-' * 62)
for depth in [1, 2, 3, 4, 5, 6, 8, 10, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(X_train, y_train)
    tr = accuracy_score(y_train, m.predict(X_train))
    te = accuracy_score(y_test,  m.predict(X_test))
    gap = tr - te
    label = depth if depth is not None else 'None (unlimited)'
    if gap > 0.10:
        status = '⚠ OVERFIT  — train >> test'
    elif tr < 0.75:
        status = '⚠ UNDERFIT — model too simple'
    else:
        status = '✓ Good fit'
    print(f'{str(label):<12} {tr:>10.4f} {te:>10.4f} {gap:>8.4f}  {status}')

print('\nKey rule: a large Train-Test gap (> ~0.10) signals overfitting.')
print('Reduce max_depth, add regularization, or get more training data to fix it.')